# Heart Disease Risk Prediction with Interpretable Machine Learning

**Author:** Chinonye Anams

Building a transparent machine learning model to predict cardiovascular risk and explain key clinical drivers using SHAP.

---

## Problem Statement

Cardiovascular disease is a leading cause of mortality worldwide.  
This project builds an interpretable ML model to predict heart disease risk and uncover the most important clinical predictors.

## Importing Library

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc

import shap

## Data Loading and Inspection

In [ ]:
df = pd.read_csv("heart_cleveland.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df['condition'].value_counts()

In [ ]:
df['condition'].value_counts(normalize=True)

In [ ]:
df.describe()

#### Data Quality Assessment

The dataset showed no missing values or inconsistent data types. All features were properly encoded as numeric variables, allowing for immediate analytical and modeling workflows without extensive preprocessing.

## Exploratory Data Analysis

We explore relationships between clinical features and heart disease to identify early signals and guide modeling.

In [ ]:
sns.boxplot(x='condition', y='age', data=df)
plt.title("Age vs Heart Disease")
plt.show()

#### Age and Cardiovascular Risk

Patients diagnosed with heart disease show a noticeably higher median age (late 50s) compared to non-diseased individuals (early 50s).

The distribution suggests an age-related risk gradient, consistent with established cardiovascular epidemiology. While heart disease is present across a wide age range, risk concentration increases in later decades of life.

This reinforces age as a meaningful — though not standalone — predictor of cardiovascular disease.

In [ ]:
sns.countplot(x='cp', hue='condition', data=df)
plt.title("Chest Pain Type vs Heart Disease")
plt.show()

#### Chest Pain Type and Diagnostic Signal

Chest pain type demonstrated one of the strongest separations between groups. The asymptomatic category (cp = 3) showed a disproportionately high association with heart disease, while non-diseased patients were more evenly distributed across other chest pain types.

This pattern highlights an important clinical reality: a significant proportion of cardiovascular disease cases may present without classic anginal symptoms, reinforcing the need for data-driven risk assessment beyond symptom-based screening.

The strength of this signal suggests that chest pain type is likely to be a dominant feature in predictive modeling.

In [ ]:
sns.boxplot(x='condition', y='thalach', data=df)
plt.title("Max Heart Rate vs Heart Disease")
plt.show()

#### Maximum Heart Rate and Cardiovascular Health

Maximum heart rate (thalach) showed a clear inverse relationship with heart disease presence. Patients diagnosed with heart disease exhibited a lower median maximum heart rate (≈140 bpm) compared to non-diseased individuals (≈160 bpm).

This finding is physiologically plausible, as cardiovascular disease can impair cardiac output and exercise tolerance, leading to reduced peak heart rate during stress or exertion.

The distribution suggests that reduced cardiovascular performance may serve as an important functional indicator of underlying heart disease.

In [ ]:
sns.countplot(x='exang', hue='condition', data=df)
plt.title("Exercise-Induced Angina vs Heart Disease")
plt.show()

#### Exercise-Induced Angina and Disease Association

Exercise-induced angina showed a strong association with heart disease. While absolute counts appeared similar across groups, proportional analysis revealed that patients reporting exercise-induced angina were far more likely to have heart disease.

In contrast, individuals without exercise-induced angina were predominantly non-diseased.

This aligns with clinical understanding, as exertional chest pain is a hallmark symptom of underlying coronary artery disease and impaired myocardial perfusion.

## Correlation Analysis

We examine linear relationships between features and heart disease to identify strong predictors.

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

Correlation analysis revealed several strong predictors of heart disease. The most positively correlated feature was thal (~0.52), suggesting a strong association between myocardial perfusion abnormalities and disease presence. The number of major vessels involved (ca, ~0.46) and ST depression (oldpeak, ~0.42) also demonstrated meaningful positive relationships with cardiovascular pathology.

Exercise-induced angina (exang) further reinforced symptom-based predictive value.

In contrast, maximum heart rate (thalach) showed a notable negative correlation (~-0.42), supporting earlier observations that reduced cardiovascular performance is associated with disease presence.

Overall, the correlation structure reflects a combination of structural, symptomatic, and functional indicators contributing to heart disease risk.

## Train-Test Split

Data is split using a stratified approach to preserve class balance and ensure reliable evaluation.

In [ ]:
X = df.drop('condition', axis=1)
y = df['condition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train.shape, X_test.shape

## Baseline Model — Logistic Regression

A logistic regression model is used as a strong, interpretable baseline suitable for healthcare applications.

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

In [ ]:
y_pred_log = log_model.predict(X_test)

## Model Evaluation

We evaluate performance using accuracy, precision, recall, and ROC-AUC to assess clinical usefulness.

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

The logistic regression model achieved an accuracy of 92% on the held-out test set, demonstrating strong predictive performance even without extensive feature engineering.

Class-level analysis revealed high precision (1.00) for heart disease predictions, indicating that positive classifications were highly reliable. However, recall for diseased patients (0.82) suggests that some true cases may be missed, highlighting the classic precision-recall trade-off in medical classification tasks.

Overall, the logistic regression model provides a strong and interpretable foundation for subsequent model comparisons.

In [ ]:
y_probs = log_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()

#### ROC Curve and Discriminative Performance

To evaluate the model’s ability to distinguish between diseased and non-diseased patients across classification thresholds, a Receiver Operating Characteristic (ROC) analysis was performed.

The logistic regression model achieved an Area Under the Curve (AUC) of 0.95, indicating excellent discriminative performance. This suggests that the model can reliably rank patients by cardiovascular risk, correctly distinguishing diseased individuals from healthy ones in the vast majority of cases.

In clinical prediction settings, such a high AUC highlights strong potential for risk stratification and decision support applications.

## Model Comparison

A Random Forest model is trained to test whether nonlinear models improve performance over logistic regression.

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Surprisingly, the Random Forest did not outperform the simpler model. While performance remained strong (88% accuracy), both recall and overall accuracy were slightly lower than logistic regression.

This suggests that the dataset may exhibit largely linear separability, where dominant clinical features such as thal and ca provide strong predictive signal without requiring complex nonlinear modeling.

This finding reinforces the value of interpretable models, as simpler approaches may achieve comparable or superior performance while remaining more transparent for clinical decision support.

## Model Explainability with SHAP

SHAP is used to understand global feature importance and ensure predictions align with clinical reasoning.

In [ ]:
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_test)

Global SHAP analysis identified chest pain type (cp) as the most influential predictor, followed by the number of major vessels (ca) and maximum heart rate (thalach). These findings are clinically meaningful, as they reflect a combination of symptomatic presentation, structural cardiovascular burden, and functional cardiac capacity.

Interestingly, SHAP results differed slightly from simple correlation analysis, elevating chest pain type as the dominant predictor. This highlights the value of model-based explainability methods in capturing nonlinear interactions and conditional feature importance that traditional statistical measures may overlook.

## Patient-Level Explanation

We analyze an individual prediction using SHAP force plots to understand how features influence risk at the patient level.

In [ ]:
patient = X_test.iloc[[0]]
patient

In [ ]:
shap.force_plot(
    explainer.expected_value[1],
    shap_values[0, :, 1],
    patient
)

For one representative patient, the model identified chest pain type (cp), resting blood pressure (trestbps), and ST depression (oldpeak) as primary contributors pushing the prediction toward heart disease. These features reflect a combination of symptomatic presentation and ischemic stress markers.

Conversely, features such as maximum heart rate (thalach), number of affected vessels (ca), and absence of exercise-induced angina (exang) exerted protective influence, counterbalancing overall risk.

## Conclusion

This project explored the development of an interpretable machine learning pipeline for cardiovascular disease prediction using structured clinical data. Through a combination of exploratory analysis, statistical reasoning, and model comparison, several consistent and clinically meaningful patterns emerged.

Key predictors of heart disease included chest pain type, number of affected vessels, ST depression, and maximum heart rate — features that collectively capture symptomatic presentation, structural disease burden, and functional cardiac capacity. Logistic regression achieved strong performance (92% accuracy, AUC = 0.95), outperforming more complex models and highlighting the effectiveness of interpretable approaches in structured clinical settings.

Importantly, explainability analysis using SHAP reinforced the clinical plausibility of model behavior. Both global and patient-level explanations demonstrated that predictions were driven by coherent physiological signals rather than opaque statistical artifacts. This level of transparency is essential for real-world healthcare applications, where trust and interpretability are critical.

Overall, this project demonstrates that carefully designed, interpretable machine learning workflows can provide meaningful insights into cardiovascular risk patterns while maintaining transparency suitable for clinical decision support. The approach highlights the value of combining domain knowledge with data-driven modeling to build trustworthy healthcare AI systems.